In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
# 1. 라이브러리 불러오기
import os
import torch
import clip
from PIL import Image, ImageFile
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau
import random
import numpy as np
from torch.nn import CrossEntropyLoss
from torch.optim import Adam

ImageFile.LOAD_TRUNCATED_IMAGES = True

#  2. CLIP 모델 불러오기
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
model.float()  # 모델 전체를 float32로 변환 (NaN 방지)


In [ ]:
#  3. 전처리 정의
train_preprocess = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
                         (0.26862954, 0.26130258, 0.27577711))
])

test_preprocess = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
                         (0.26862954, 0.26130258, 0.27577711))
])

In [ ]:
# 4. 데이터셋 클래스 정의
class CLIPEmotionDataset(Dataset):
    def __init__(self, image_dir, preprocess=None):
        self.data = []
        self.label_to_index = {}
        self.index_to_label = []
        self.preprocess = preprocess

        for label in sorted(os.listdir(image_dir)):
            label_path = os.path.join(image_dir, label)
            if os.path.isdir(label_path):
                if label not in self.label_to_index:
                    self.label_to_index[label] = len(self.label_to_index)
                    self.index_to_label.append(label)
                for fname in os.listdir(label_path):
                    if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                        full_path = os.path.join(label_path, fname)
                        self.data.append((full_path, self.label_to_index[label]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# Transform 따로 적용
class TransformingDataset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        path, label = self.base_dataset[self.indices[idx]]
        try:
            image = self.transform(Image.open(path).convert("RGB"))
            return image, label
        except:
            return torch.zeros((3, 224, 224)), -1


In [ ]:
# 5. 데이터 분할
image_dir = "/content/drive/MyDrive/emotions"
full_dataset = CLIPEmotionDataset(image_dir)

indices = list(range(len(full_dataset)))
random.shuffle(indices)

train_end = int(0.75 * len(indices))
val_end = int(0.9 * len(indices))

train_idx = indices[:train_end]
val_idx = indices[train_end:val_end]
test_idx = indices[val_end:]

print(f"총 이미지 수: {len(full_dataset)}")
print(f"Train: {len(train_idx)}장")
print(f"Validation: {len(val_idx)}장")
print(f"Test: {len(test_idx)}장")


# 증강 포함된 Dataset 인스턴스 만들기
train_loader = DataLoader(
    TransformingDataset(full_dataset, train_idx, train_preprocess),
    batch_size=64, shuffle=True, num_workers=2
)

val_loader = DataLoader(
    TransformingDataset(full_dataset, val_idx, test_preprocess),
    batch_size=64, shuffle=False, num_workers=2
)

test_loader = DataLoader(
    TransformingDataset(full_dataset, test_idx, test_preprocess),
    batch_size=64, shuffle=False, num_workers=2
)


In [ ]:
# 6. 텍스트 프롬프트 만들기
label_names = full_dataset.label_to_index.keys()
text_tokens = clip.tokenize([f"a {label} image" for label in label_names]).to(device)


In [ ]:

# ✅ Feature 추출 함수 정의 (1회 실행)
def extract_features_and_save(loader, model, save_path, device):
    model.eval()
    features, labels = [], []
    with torch.no_grad():
        for images, label_batch in tqdm(loader, desc=f"Extracting to {save_path}"):
            valid = label_batch != -1
            images, label_batch = images[valid].to(device), label_batch[valid].to(device)
            feats = model.encode_image(images).float()
            feats /= (feats.norm(dim=-1, keepdim=True) + 1e-6)
            features.append(feats.cpu())
            labels.append(label_batch.cpu())
    torch.save((torch.cat(features), torch.cat(labels)), save_path)

# FeatureDataset
class FeatureDataset(Dataset):
    def __init__(self, path):
        self.features, self.labels = torch.load(path)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# features 추출 (한 번만 실행)
extract_features_and_save(train_loader, model, "train_feats.pt", device)
extract_features_and_save(val_loader, model, "val_feats.pt", device)
extract_features_and_save(test_loader, model, "test_feats.pt", device)

# Feature 기반 학습용 로더
train_feat_loader = DataLoader(FeatureDataset("train_feats.pt"), batch_size=64, shuffle=True)
val_feat_loader = DataLoader(FeatureDataset("val_feats.pt"), batch_size=64)

# 텍스트 feature 고정
with torch.no_grad():
    text_features = model.encode_text(text_tokens).float().to(device)
    text_features /= (text_features.norm(dim=-1, keepdim=True) + 1e-6)


In [ ]:
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

model_dir = "/content/drive/MyDrive/models"
os.makedirs(model_dir, exist_ok=True)
weight_path = os.path.join(model_dir, 'emotion_clip_weights.pth')

# Classifier 정의 (MLP)
class MLPClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.mlp = torch.nn.Sequential(
           nn.Linear(in_dim, hidden_dim),
           nn.LayerNorm(hidden_dim),
           nn.ReLU(),
           nn.Dropout(0.5),
           nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        return self.mlp(x)

feat_dim = train_feat_loader.dataset[0][0].shape[0]
num_classes = len(label_names)
classifier = MLPClassifier(feat_dim, 256, num_classes).to(device)

# 8. 학습 설정
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
loss_fn = CrossEntropyLoss()  # 🪄 label smoothing 적용

train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

#  학습 루프
epochs = 20
best_val_acc = 0.0

for epoch in range(epochs):
    print(f"\n🔁 Epoch {epoch+1}/{epochs}")
    classifier.train()
    total_loss, correct, total = 0, 0, 0
    for feats, labels in tqdm(train_feat_loader, desc="Train"):
        feats, labels = feats.to(device), labels.to(device)
        logits = classifier(feats)
        loss = loss_fn(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * feats.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += feats.size(0)
    train_loss = total_loss / total
    train_acc = correct / total

    # 🔍 Validation
    classifier.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for feats, labels in tqdm(val_feat_loader, desc="Val"):
            feats, labels = feats.to(device), labels.to(device)
            logits = classifier(feats)
            loss = loss_fn(logits, labels)
            total_loss += loss.item() * feats.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += feats.size(0)
    val_loss = total_loss / total
    val_acc = correct / total

    print(f"📉 Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"📊 Val   Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(classifier.state_dict(), weight_path)
        print("✅ Best model saved!")

    scheduler.step(epoch)

In [ ]:
# 9. 시각화
plt.figure(figsize=(12, 5))

# Loss 그래프
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Val Loss', marker='x')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over Epochs")
plt.legend()
plt.grid(True)

# Accuracy 그래프
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Acc', marker='o')
plt.plot(val_accuracies, label='Val Acc', marker='x')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy over Epochs")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# 10. 테스트 정확도 평가
test_feat_loader = DataLoader(FeatureDataset("test_feats.pt"), batch_size=64)
classifier.load_state_dict(torch.load("best_classifier.pth"))
classifier.eval()
correct, total = 0, 0
with torch.no_grad():
    for feats, labels in test_feat_loader:
        feats, labels = feats.to(device), labels.to(device)
        logits = classifier(feats)
        correct += (logits.argmax(1) == labels).sum().item()
        total += feats.size(0)
test_acc = correct / total
print(f"🎯 Test Accuracy: {test_acc:.4f}")

In [ ]:
# 11. 학습 종료 후 모델 저장

model_dir = "/content/drive/MyDrive/models"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'emotion_clip.pth')
torch.save(model, model_path)
print(f"모델이 저장되었습니다: {model_path}")
